# Point-in-Time Multi-Frequency Wasserstein Signal — Validation and Publication

**Assertion barrier for the signal inputs used in the article.**

This notebook validates the candidate and matrix-audit Parquet files produced by `notebooks/05_compute_pit_signals.ipynb`. It checks the full point-in-time contract, configuration coverage, exact universe membership, common asset axes across frequencies, absence of imputation, numerical invariants, cache identity, and an independent deterministic recomputation.

Publication occurs only in the final section and only after every preceding assertion passes. The canonical validated dataset and the compatibility series consumed by `notebooks/01_signal.ipynb` are written atomically.

## 1. Setup and publication policy

The candidate cache is never treated as an article input. `PUBLISH=True` authorises publication only within this notebook and only after the complete validation sequence has executed successfully in the current kernel.

In [ ]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
import polars as pl

ROOT = Path.cwd()
if ROOT.name in {'notebooks', 'tests'}:
    ROOT = ROOT.parent

SOURCES = {
    'big_caps': ROOT / 'data' / 'processed' / 'nyse_big_caps_pit_daily.parquet',
    'small_caps_p20_p50': ROOT / 'data' / 'processed' / 'nyse_small_caps_p20_p50_pit_daily.parquet',
}
CANDIDATE_PATH = ROOT / 'cache' / 'signal' / 'rho_pit_candidate.parquet'
AUDIT_PATH = ROOT / 'cache' / 'signal' / 'rho_pit_matrix_audit.parquet'
SIGNAL_DIR = ROOT / 'data' / 'signals'
CANONICAL_PATH = SIGNAL_DIR / 'rho_pit_validated.parquet'
MANIFEST_PATH = SIGNAL_DIR / 'rho_pit_validation_manifest.parquet'

EXPECTED_ENGINE = 'pit-signal-v1.1.0'
EXPECTED_CONFIGS = {
    'reference_w36', 'reference_w48', 'reference_w60',
    'cardinality_m36', 'cardinality_m72',
    'weights_tk', 'weights_log_tk', 'scaling_vol',
    'scaling_h04', 'scaling_h06', 'distance_exact', 'barycenter_1d',
}
PUBLISH = True

for path in [*SOURCES.values(), CANDIDATE_PATH, AUDIT_PATH]:
    assert path.exists(), path

def atomic_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    df.to_parquet(temporary, index=False)
    temporary.replace(path)

def source_signature(path: Path) -> str:
    stat = path.stat()
    payload = f'{path.resolve()}|{stat.st_size}|{stat.st_mtime_ns}'
    return hashlib.sha256(payload.encode()).hexdigest()

def asset_hash(assets) -> str:
    return hashlib.sha256(','.join(map(str, assets)).encode()).hexdigest()

def seed_for(universe: str, formation_month: pd.Timestamp, base_seed=20250301) -> int:
    key = f'{base_seed}|{universe}|{formation_month:%Y-%m}'
    return int(hashlib.sha256(key.encode()).hexdigest()[:8], 16)

print('Validation inputs found. No publication has occurred.')

## 2. Candidate schema and numerical invariants

Each universe–configuration–date key must be unique. The signal is a finite non-negative squared distance, `sqrt_rho` must be its numerical square root, frequency weights must be strictly positive and sum to one, and the nullable exponent is permitted only for realized-volatility scaling.

In [ ]:
candidate = pd.read_parquet(CANDIDATE_PATH)
audit = pd.read_parquet(AUDIT_PATH)
for col in ['date', 'formation_month']:
    candidate[col] = pd.to_datetime(candidate[col])
    audit[col] = pd.to_datetime(audit[col])

SIGNAL_REQUIRED = {
    'date', 'formation_month', 'universe', 'config_id', 'lookback_months', 'n_assets',
    'M_atoms', 'n_projections', 'n_quantiles', 'frequency_weights', 'scaling',
    'exponent', 'distance', 'barycenter', 'seed', 'lambda_daily', 'lambda_weekly',
    'lambda_monthly', 'rho', 'sqrt_rho', 'engine_version', 'source_signature', 'config_digest',
}
AUDIT_REQUIRED = {
    'date', 'formation_month', 'universe', 'config_id', 'start_date', 'window_end',
    'n_assets', 'n_daily', 'n_weekly', 'n_monthly', 'daily_nulls', 'weekly_nulls',
    'monthly_nulls', 'same_columns', 'no_future_observations', 'asset_hash',
    'engine_version', 'source_signature', 'config_digest',
}
assert not (SIGNAL_REQUIRED - set(candidate.columns)), sorted(SIGNAL_REQUIRED - set(candidate.columns))
assert not (AUDIT_REQUIRED - set(audit.columns)), sorted(AUDIT_REQUIRED - set(audit.columns))
assert len(candidate) > 0 and len(audit) > 0
assert not candidate.duplicated(['universe', 'config_id', 'date']).any()
assert not audit.duplicated(['universe', 'config_id', 'date']).any()
assert candidate['date'].notna().all() and candidate['formation_month'].notna().all()
assert candidate['universe'].isin(SOURCES).all()
assert candidate['engine_version'].eq(EXPECTED_ENGINE).all()
assert candidate['config_digest'].str.fullmatch(r'[0-9a-f]{64}').all()
assert np.isfinite(candidate['rho']).all() and candidate['rho'].ge(0).all()
assert np.isfinite(candidate['sqrt_rho']).all() and candidate['sqrt_rho'].ge(0).all()
assert np.allclose(candidate['sqrt_rho'] ** 2, candidate['rho'], rtol=1e-12, atol=1e-15)

lambda_cols = ['lambda_daily', 'lambda_weekly', 'lambda_monthly']
assert np.isfinite(candidate[lambda_cols].to_numpy()).all()
assert candidate[lambda_cols].gt(0).all().all()
assert np.allclose(candidate[lambda_cols].sum(axis=1), 1.0, rtol=0, atol=1e-12)
assert candidate.loc[candidate['scaling'].ne('realized_volatility'), 'exponent'].notna().all()
assert candidate.loc[candidate['scaling'].eq('realized_volatility'), 'exponent'].isna().all()
assert candidate['n_assets'].eq(100).all()
assert candidate['n_projections'].gt(0).all() and candidate['n_quantiles'].gt(1).all()

print(f'Candidate schema and numerical invariants: PASS ({len(candidate):,} rows).')

## 3. Configuration coverage and cache identity

Every configuration must cover exactly the same formation dates within a universe. A configuration may change the estimator or window length, but it may not silently change the monthly investment universe. Source fingerprints and configuration digests must agree between candidate and audit artifacts.

In [ ]:
assert set(candidate['universe']) == set(SOURCES)
for universe in SOURCES:
    sub = candidate[candidate['universe'].eq(universe)]
    assert set(sub['config_id']) == EXPECTED_CONFIGS
    reference_dates = set(sub.loc[sub['config_id'].eq('reference_w36'), 'date'])
    assert reference_dates
    for config_id, group in sub.groupby('config_id'):
        assert set(group['date']) == reference_dates, f'{universe}/{config_id}: date coverage differs'
        assert group['date'].is_monotonic_increasing
        assert group['config_digest'].nunique() == 1

expected_signatures = {u: source_signature(path) for u, path in SOURCES.items()}
for universe, signature in expected_signatures.items():
    assert candidate.loc[candidate['universe'].eq(universe), 'source_signature'].eq(signature).all()
    assert audit.loc[audit['universe'].eq(universe), 'source_signature'].eq(signature).all()

keys = ['universe', 'config_id', 'date']
joined = candidate[keys + ['config_digest']].merge(
    audit[keys + ['config_digest']], on=keys, how='outer', suffixes=('_signal', '_audit'), indicator=True
)
assert joined['_merge'].eq('both').all()
assert joined['config_digest_signal'].eq(joined['config_digest_audit']).all()

print('Configuration coverage and cache identity: PASS.')

## 4. Matrix alignment and no-imputation assertions

A valid row has zero missing values at every frequency, identical ordered PERMNO columns, an observation cutoff no later than the formation close, and one common asset hash across all robustness configurations for the same universe-date. This is the central correction to the former hybrid-matrix construction.

In [ ]:
for col in ['start_date', 'window_end']:
    audit[col] = pd.to_datetime(audit[col])

assert audit[['daily_nulls', 'weekly_nulls', 'monthly_nulls']].eq(0).all().all()
assert audit['same_columns'].eq(True).all()
assert audit['no_future_observations'].eq(True).all()
assert audit['window_end'].le(audit['date']).all()
assert audit['n_daily'].gt(0).all() and audit['n_weekly'].gt(0).all() and audit['n_monthly'].gt(0).all()
assert audit['asset_hash'].str.fullmatch(r'[0-9a-f]{64}').all()
assert audit.groupby(['universe', 'date'])['asset_hash'].nunique().eq(1).all()
assert audit.groupby(['universe', 'date'])['n_assets'].nunique().eq(1).all()

metadata_check = candidate[keys + ['n_assets']].merge(
    audit[keys + ['n_assets']], on=keys, suffixes=('_signal', '_audit'), validate='one_to_one'
)
assert metadata_check['n_assets_signal'].eq(metadata_check['n_assets_audit']).all()

print('Matrix alignment and no-imputation assertions: PASS.')

## 5. Full PIT membership reconciliation

Membership is reconstructed directly from each processed source. For every formation month, active month must equal $M+1$, all selected securities must satisfy the strict 60-month history flag, ranks must be unique and bounded by 100, and the reconstructed count and PERMNO hash must match every calculated configuration.

In [ ]:
membership_frames = {}
for universe, path in SOURCES.items():
    membership = (
        pl.scan_parquet(path)
        .filter(pl.col('is_formation_member') == 1)
        .select([
            pl.col('formation_member_month').alias('formation_month'),
            pl.col('formation_member_active_month').alias('active_month'),
            'PERMNO', pl.col('formation_member_rank').alias('rank'),
            pl.col('formation_member_strict_hist60').alias('strict_hist60'),
        ])
        .collect(engine='streaming')
        .to_pandas()
    )
    membership['formation_month'] = pd.to_datetime(membership['formation_month'])
    membership['active_month'] = pd.to_datetime(membership['active_month'])
    assert membership['formation_month'].notna().all()
    assert membership['active_month'].eq(membership['formation_month'] + pd.offsets.MonthBegin(1)).all()
    assert membership['strict_hist60'].eq(True).all()
    assert not membership.duplicated(['formation_month', 'PERMNO']).any()
    assert not membership.duplicated(['formation_month', 'rank']).any()
    assert membership['rank'].between(1, 100).all()
    assert membership.groupby('formation_month')['PERMNO'].size().eq(100).all()

    reconstructed = []
    for formation_month, group in membership.groupby('formation_month', sort=True):
        assets = group.sort_values(['rank', 'PERMNO'])['PERMNO'].astype(int).tolist()
        reconstructed.append(dict(
            universe=universe, formation_month=formation_month,
            n_assets_source=len(assets), asset_hash_source=asset_hash(assets),
        ))
    reconstructed = pd.DataFrame(reconstructed)
    reference = audit.loc[
        audit['universe'].eq(universe) & audit['config_id'].eq('reference_w36'),
        ['formation_month', 'n_assets', 'asset_hash'],
    ]
    check = reference.merge(reconstructed, on='formation_month', how='outer', validate='one_to_one', indicator=True)
    assert check['_merge'].eq('both').all()
    assert check['n_assets'].eq(check['n_assets_source']).all()
    assert check['asset_hash'].eq(check['asset_hash_source']).all()
    membership_frames[universe] = membership

print('Full PIT membership reconciliation: PASS.')

## 6. Independent deterministic recomputation

For the first, middle, and last available formation dates of each universe, this section independently rebuilds the 36-month daily panel from the source, compounds weekly and monthly returns, applies square-root-of-horizon scaling, and recomputes the 1-D quantile-barycenter signal. Exact agreement verifies the membership, window boundary, aggregation, scaling, random seed, and signal-value pipeline jointly.

In [ ]:
N_PROJECTIONS = int(candidate['n_projections'].iloc[0])
N_QUANTILES = int(candidate['n_quantiles'].iloc[0])
assert candidate['n_projections'].eq(N_PROJECTIONS).all()
assert candidate['n_quantiles'].eq(N_QUANTILES).all()

def independent_rho_1d(arrays, seed):
    lam = np.ones(3) / 3.0
    rng = np.random.default_rng(seed)
    q = np.linspace(0.0, 1.0, N_QUANTILES)
    total = 0.0
    dimension = arrays[0].shape[1]
    for _ in range(N_PROJECTIONS):
        direction = rng.standard_normal(dimension)
        direction /= np.linalg.norm(direction)
        quantiles = [np.quantile(x @ direction, q) for x in arrays]
        barycenter = sum(lam[k] * quantiles[k] for k in range(3))
        total += sum(lam[k] * np.mean((quantiles[k] - barycenter) ** 2) for k in range(3))
    return float(total / N_PROJECTIONS)

recomputed_rows = []
for universe, path in SOURCES.items():
    target = candidate[
        candidate['universe'].eq(universe) & candidate['config_id'].eq('barycenter_1d')
    ].sort_values('formation_month').reset_index(drop=True)
    sample_positions = sorted(set([0, len(target) // 2, len(target) - 1]))

    for position in sample_positions:
        row = target.iloc[position]
        formation_month = pd.Timestamp(row['formation_month'])
        formation_date = pd.Timestamp(row['date'])
        start_month = formation_month - pd.DateOffset(months=35)
        member = membership_frames[universe]
        assets = (member.loc[member['formation_month'].eq(formation_month)]
                        .sort_values(['rank', 'PERMNO'])['PERMNO'].astype(int).tolist())

        window = (
            pl.scan_parquet(path)
            .filter(
                pl.col('PERMNO').is_in(assets),
                pl.col('DlyCalDt') >= start_month.to_pydatetime(),
                pl.col('DlyCalDt') <= formation_date.to_pydatetime(),
            )
            .select(['DlyCalDt', 'PERMNO', 'DlyRet'])
            .collect(engine='streaming')
            .to_pandas()
        )
        window['DlyCalDt'] = pd.to_datetime(window['DlyCalDt'])
        assert not window.duplicated(['DlyCalDt', 'PERMNO']).any()
        daily = window.pivot(index='DlyCalDt', columns='PERMNO', values='DlyRet').sort_index()
        daily = daily.reindex(columns=assets)
        assert daily.notna().all().all()
        observed_months = daily.index.to_period('M').unique().sort_values()
        expected_months = pd.period_range(start_month, formation_month, freq='M')
        assert observed_months.equals(expected_months)
        assert daily.index.max() == formation_date

        weekly = (1.0 + daily).resample('W-FRI').prod(min_count=1) - 1.0
        monthly = (1.0 + daily).resample('ME').prod(min_count=1) - 1.0
        assert list(daily.columns) == list(weekly.columns) == list(monthly.columns)
        arrays = [daily.to_numpy(float), weekly.to_numpy(float) / np.sqrt(5.0),
                  monthly.to_numpy(float) / np.sqrt(21.0)]
        observed = independent_rho_1d(arrays, seed_for(universe, formation_month))
        assert np.isclose(observed, row['rho'], rtol=1e-11, atol=1e-15), (universe, formation_month, observed, row['rho'])
        recomputed_rows.append(dict(universe=universe, date=formation_date, candidate=row['rho'], recomputed=observed))

recomputed = pd.DataFrame(recomputed_rows)
display(recomputed)
print('Independent deterministic recomputation: PASS.')

## 7. Cross-series consistency

Reference and robustness series must remain genuine monthly time series. Dates are unique and ordered, the formation timestamp belongs to the same calendar month as the publication date, and every series has non-zero time variation. These checks catch silent duplicates, stale concatenations, or degenerate estimator output.

In [ ]:
assert candidate['date'].dt.to_period('M').eq(candidate['formation_month'].dt.to_period('M')).all()
for (universe, config_id), group in candidate.groupby(['universe', 'config_id']):
    group = group.sort_values('date')
    periods = group['date'].dt.to_period('M')
    assert periods.is_unique
    assert periods.is_monotonic_increasing
    assert group['rho'].nunique() > 1, f'Degenerate series: {universe}/{config_id}'
    assert group['rho'].std(ddof=1) > 0

reference = candidate[candidate['config_id'].eq('reference_w36')]
assert reference.groupby('universe')['rho'].count().gt(24).all()
print('Cross-series consistency: PASS.')

## 8. Validated publication

The canonical long-form Parquet preserves both universes and all configurations. Compatibility files contain only `date` and `rho`, matching the loader contract already used by `01_signal.ipynb`. `V_uni.parquet` is the validated big-cap reference signal; robustness aliases replace the former independently-cleaned calculations. `V_small_uni.parquet` exposes the validated NYSE P20–P50 reference signal.

In [ ]:
ALIAS_MAP = {
    'V_uni.parquet': ('big_caps', 'reference_w36'),
    'V_uni_W48.parquet': ('big_caps', 'reference_w48'),
    'V_uni_W60.parquet': ('big_caps', 'reference_w60'),
    'U_M36.parquet': ('big_caps', 'cardinality_m36'),
    'U_M72.parquet': ('big_caps', 'cardinality_m72'),
    'U_lamTk.parquet': ('big_caps', 'weights_tk'),
    'U_lamlog.parquet': ('big_caps', 'weights_log_tk'),
    'U_volscale.parquet': ('big_caps', 'scaling_vol'),
    'U_H04.parquet': ('big_caps', 'scaling_h04'),
    'U_H06.parquet': ('big_caps', 'scaling_h06'),
    'U_exact.parquet': ('big_caps', 'distance_exact'),
    'U_bary1d.parquet': ('big_caps', 'barycenter_1d'),
    'V_small_uni.parquet': ('small_caps_p20_p50', 'reference_w36'),
}

# These objects exist only if every preceding validation section ran successfully.
required_validation_objects = {
    'candidate', 'audit', 'joined', 'metadata_check', 'membership_frames',
    'recomputed', 'reference', 'expected_signatures',
}
missing_validation_objects = required_validation_objects - set(globals())
assert not missing_validation_objects, f'Validation cells were skipped: {sorted(missing_validation_objects)}'
assert len(recomputed) == 6
assert joined['_merge'].eq('both').all()
assert metadata_check['n_assets_signal'].eq(metadata_check['n_assets_audit']).all()
assert set(membership_frames) == set(SOURCES)

if PUBLISH:
    SIGNAL_DIR.mkdir(parents=True, exist_ok=True)
    canonical = candidate.sort_values(['universe', 'config_id', 'date']).reset_index(drop=True)
    atomic_parquet(canonical, CANONICAL_PATH)

    alias_rows = []
    for filename, (universe, config_id) in ALIAS_MAP.items():
        series = (canonical.loc[
            canonical['universe'].eq(universe) & canonical['config_id'].eq(config_id),
            ['date', 'rho'],
        ].sort_values('date').reset_index(drop=True))
        assert len(series) > 0 and not series['date'].duplicated().any()
        output_path = SIGNAL_DIR / filename
        atomic_parquet(series, output_path)
        alias_rows.append(dict(filename=filename, universe=universe, config_id=config_id, rows=len(series),
                               date_min=series['date'].min(), date_max=series['date'].max()))

    manifest = (
        canonical.groupby(['universe', 'config_id', 'engine_version', 'source_signature', 'config_digest'], as_index=False)
                 .agg(rows=('rho', 'size'), date_min=('date', 'min'), date_max=('date', 'max'))
                 .assign(validation_passed=True, canonical_file=CANONICAL_PATH.name)
    )
    atomic_parquet(manifest, MANIFEST_PATH)
    print('Published canonical:', CANONICAL_PATH)
    print('Published manifest: ', MANIFEST_PATH)
    display(pd.DataFrame(alias_rows))
else:
    print('PUBLISH=False — all assertions passed; no files were written.')

## 9. Handoff to the article notebook

`notebooks/01_signal.ipynb` may now load `data/signals/V_uni.parquet` through its existing `load_signal` function. Its robustness loaders can reuse the published `U_*.parquet` aliases, so those figures no longer need to reconstruct daily, weekly, and monthly matrices internally. The canonical `rho_pit_validated.parquet` remains the auditable source of truth for both the big-cap and P20–P50 series.